In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [6]:
data = pd.read_csv("scraped.csv", encoding='latin-1')
data.head(3)

,openalex_id,title,doi,abstract,publish_time,authors,journal,pmcid,pubmed_id
0,https://openalex.org/W2110829397,PROBLEMS OF SPACE BIOLOGY,NaN,Space biology - effect of radiation exposure a...,1963-03-25,N. M. Sisakyan,Defense Technical Information Center (DTIC),NaN,NaN
1,https://openalex.org/W2035753075,Chemical space and biology,https://doi.org/10.1038/nature03192,Chemical space--which encompasses all possible...,2004-12-15,Christopher M. Dobson,Nature,NaN,https://pubmed.ncbi.nlm.nih.gov/15602547
2,https://openalex.org/W2988387100,Space biology and medicine.,NaN,Perhaps one of the greatest gifts that has bee...,1979-10-01,Arnauld Nicogossian; Stanley R. Mohler; Ð. Ð...,PubMed,NaN,https://pubmed.ncbi.nlm.nih.gov/11902165


In [7]:
df_csv = data
df_csv = df_csv[['openalex_id','title','doi','abstract','publish_time','authors','journal','doi','pmcid','pubmed_id']]
df_csv.head()

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi,pmcid,pubmed_id
0,https://openalex.org/W2110829397,PROBLEMS OF SPACE BIOLOGY,NaN,Space biology - effect of radiation exposure a...,1963-03-25,N. M. Sisakyan,Defense Technical Information Center (DTIC),NaN,NaN,NaN
1,https://openalex.org/W2035753075,Chemical space and biology,https://doi.org/10.1038/nature03192,Chemical space--which encompasses all possible...,2004-12-15,Christopher M. Dobson,Nature,https://doi.org/10.1038/nature03192,NaN,https://pubmed.ncbi.nlm.nih.gov/15602547
2,https://openalex.org/W2988387100,Space biology and medicine.,NaN,Perhaps one of the greatest gifts that has bee...,1979-10-01,Arnauld Nicogossian; Stanley R. Mohler; Ð. Ð...,PubMed,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/11902165
3,https://openalex.org/W3114753755,Advantages and Limitations of Current Microgra...,https://doi.org/10.3390/app11010068,Human Space exploration has created new challe...,2020-12-23,Francesca Ferranti; Marta Del Bianco; Claudia ...,Applied Sciences,https://doi.org/10.3390/app11010068,NaN,NaN
4,https://openalex.org/W371744478,AN UPDATE ON PLANT SPACE BIOLOGY,NaN,Space craft in low-Earth orbit can provide a m...,2011-08-30,Chris Wolverton; John Z. Kiss,Digital Commons - OWU (Ohio Wesleyan University),NaN,NaN,NaN


In [8]:
print(len(df_csv))
df_csv = df_csv[~df_csv['abstract'].isnull()]
print(len(df_csv))

199993
199989


In [9]:
df_csv['abstract'] = df_csv['abstract'].apply(lambda x:
                                          x.replace('BACKGROUND:','').replace('BACKGROUNDS:','').replace('OBJECTIVES:','')
                                          .replace('OBJECTIVE:','').replace('METHODS:','').replace('METHOD:','')
                                          .replace('RESULTS:','').replace('RESULT:','')
                                          .replace('CONCLUSION:','').replace('CONCLUSIONS:',''))

In [10]:
df_csv['abstract'] = df_csv['abstract'].apply(lambda x: x.lower())
df_csv['abstract'] = df_csv['abstract'].apply(lambda x: x.replace('this article is protected by copyright. all rights reserved',''))

In [11]:
df_csv.to_csv('abstract_final.csv', index=None)

In [12]:
print(len(df_csv))
df_csv = df_csv[~df_csv['publish_time'].isnull()]
print(len(df_csv))

199989
199921


In [13]:
df = df_csv
df['publish_time_new'] = pd.to_datetime(df['publish_time'], format='%Y-%m-%d',errors='coerce')

In [14]:
print(len(df))
df = df[df.publish_time_new != "NaT"]
print(len(df))

199921
199921


In [ ]:
#OPTIONAL
import datetime
df= df[df['publish_time_new']>'2020-01-01']
len(df)

18864

In [15]:
# IF NOT USED: USE LANGUAGE = MULTILINGUAL IN TRAINING THE MODEL
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def langdet (x):
    try:
        return detect(x)
    except:
        return "NA"
df['lang'] = df['abstract'].apply(lambda x: langdet(x))
df = df[df['lang'].str.contains('en')]
df.to_csv('articles_clean_text_eng.csv', index=None)
len(df)

199102

In [100]:
import pandas as pd
df = pd.read_csv('articles_clean_text_eng.csv')

In [ ]:
df1 = df
len(df1) # MUST BE SAME AS THE PREVIOUS STEP

99640

In [130]:
unk = """'Unknown'
'Not Known'
'Little is known'
'Unrevealed'
'Uncertain'
'Undetermined'
'Understudied'
'Unexplored'
'Unclear'
'Not fully understood'
'Literature gap'
'Research gap'
'Knowledge gap'
'Future studies'
'Future research'
'Research problem'
'More studies'
'More research'
'Further studies'
'Further research'"""
unk = unk.replace("\n",",").replace("'","").lower().split(",")
unk= "|".join(unk)
unk

'unknown|not known|little is known|unrevealed|uncertain|undetermined|understudied|unexplored|unclear|not fully understood|literature gap|research gap|knowledge gap|future studies|future research|research problem|more studies|more research|further studies|further research'

In [131]:
df2 = df1[df1.abstract.str.contains(unk)]
len(df2)

12816

In [4]:
# Commented lines allow for local stopword database.
import re
import nltk
import string
from textblob import TextBlob
# nltk.download('stopwords')
# nltk.download('wordnet')
stopword = nltk.corpus.stopwords.words('english')
#my_file = open("stopwords.txt", "r")
#content = my_file.read().split('\n')
#stopword.extend(content)
stopword = list(set(stopword))
stopword = [w.strip() for w in stopword]
stopword = set(stopword)
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()
from nltk import bigrams, trigrams

def removeWeirdChars(text):
    weirdPatterns = re.compile("["u"\U0001F600-\U0001F64F"u"\U0001F300-\U0001F5FF"u"\U0001F680-\U0001F6FF"u"\U0001F1E0-\U0001F1FF"u"\U00002702-\U000027B0"u"\U000024C2-\U0001F251"u"\U0001f926-\U0001f937"u'\U00010000-\U0010ffff'
                               u"\u200d"u"\u2640-\u2642"u"\u2600-\u2B55"u"\u23cf"u"\u23e9"u"\u231a"u"\u3030"u"\ufe0f"u"\u2069"u"\u2066"u"\u200c"u"\u2068"u"\u2067""]+", flags=re.UNICODE)
    return weirdPatterns.sub(r'', text)
def remove_punct(text):
    text  = "".join([char for char in text if char not in string.punctuation])
    text = re.sub('[0-9]+', '', text)
    return text
def tokenization(text):
    text = text.split()#re.split('\W+', text)
    text = ','.join(set(text))
    return text
def remove_stopwords(text):
    text = [word.strip() for word in text.split() if word not in stopword]
    text = ' '.join(text)
    return text
def stemming(text): # This is unused
    text = [ps.stem(word) for word in text.split()]
    text = ' '.join(text)
    return text

def lemmatizer(text): # This is unused
    text = [wn.lemmatize(word) for word in text.split()]
    text = ' '.join(text)
    return text
def clean_text(text):
    text_lc = " ".join([word.lower() for word in text.split() if word not in string.punctuation]) # remove puntuation
    text_rc = re.sub('[0-9]+', '', text_lc)
    tokens = re.split('\W+', text_rc)    # tokenization
    text = [word for word in tokens if word not in stopword]  # remove stopwords and stemming
    text = ' '.join(text)
    return text


df2['clean_text'] = df2['abstract'].apply(lambda x: clean_text(x))

NameError: name 'df2' is not defined

In [3]:
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')

wl = unk.split("|")
def sentence(x):
    s, w = [], []
    sent = sent_tokenize(x)
    if len(sent)<=1:
        s, w =["-1"], ["-1"]
    else:
        for c, i in enumerate(sent):
            for j in wl:
                if j in i:
                    w.append(j)
                    if c==0 or c==len(sent)-1:
                        s.append(i)
                    else:
                        s.append(sent[c-1])
                        s.append(sent[c])
                        s.append(sent[c+1])
    s = list(dict.fromkeys(s))
    w = list(dict.fromkeys(w))
    C = ".".join(s)
    D = ",".join(w)
    return pd.Series([C, D])

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shaur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


NameError: name 'unk' is not defined

In [140]:
df2[['sentences', 'words']] = df2['abstract'].apply(sentence)

In [141]:
df2.head()

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,clean_text,sentences,words
6,https://openalex.org/W2160719116,Synthetic biology for the directed evolution o...,https://doi.org/10.1039/c4cs00351a,the amino acid sequence of a protein affects b...,2014-12-15,Andrew Currin; Neil Swainston; Philip J. Day; ...,Chemical Society Reviews,https://doi.org/10.1039/c4cs00351a,NaN,https://pubmed.ncbi.nlm.nih.gov/25503938,2014-12-15,en,amino acid sequence protein affects structure ...,enzymologists distinguish binding (kd) and cat...,unknown
8,https://openalex.org/W3016143870,Space Radiation Biology for âLiving in Spaceâ,https://doi.org/10.1155/2020/4703286,space travel has advanced significantly over t...,2020-01-01,Satoshi Furukawa; Aiko Nagamatsu; Mitsuru Neno...,BioMed Research International,https://doi.org/10.1155/2020/4703286,NaN,https://pubmed.ncbi.nlm.nih.gov/32337251,2020-01-01,en,space travel advanced significantly last six d...,future research that furthers our understandin...,future research
26,https://openalex.org/W1424654272,Planning Algorithms,https://doi.org/10.1017/cbo9780511546877,planning algorithms are impacting technical di...,2006-05-29,Steven M. LaValle,Cambridge University Press eBooks,https://doi.org/10.1017/cbo9780511546877,NaN,NaN,2006-05-29,en,planning algorithms impacting technical discip...,the treatment is centered on robot motion plan...,uncertain
28,https://openalex.org/W4366735078,Perspectives for plant biology in space and an...,https://doi.org/10.1038/s41526-023-00315-x,advancements in plant space biology are requir...,2023-08-21,Veronica De Micco; Giovanna Aronne; Nicol Capl...,npj Microgravity,https://doi.org/10.1038/s41526-023-00315-x,NaN,https://pubmed.ncbi.nlm.nih.gov/37604914,2023-08-21,en,advancements plant space biology required real...,this apparent paradox is a current research ch...,knowledge gap
58,https://openalex.org/W2056901756,Plant biology in space: recent accomplishments...,https://doi.org/10.1111/plb.12127,gravity has shaped the evolution of life since...,2013-12-23,GÃ¼nter Ruyters; Markus Braun,Plant Biology,https://doi.org/10.1111/plb.12127,NaN,https://pubmed.ncbi.nlm.nih.gov/24373009,2013-12-23,en,gravity shaped evolution life since origin how...,a chapter summarising major scientific breakth...,future research


In [142]:
len(df2),len(df2.pubmed_id==None)

(12816, 12816)

In [143]:
df2.to_csv("articles_with_sentences and words1.csv", index=None)

# Data


In [144]:
import pandas as pd
df2 = pd.read_csv('articles_with_sentences and words1.csv')

In [145]:
df2.head(2)

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,clean_text,sentences,words
0,https://openalex.org/W2160719116,Synthetic biology for the directed evolution o...,https://doi.org/10.1039/c4cs00351a,the amino acid sequence of a protein affects b...,2014-12-15,Andrew Currin; Neil Swainston; Philip J. Day; ...,Chemical Society Reviews,https://doi.org/10.1039/c4cs00351a,NaN,https://pubmed.ncbi.nlm.nih.gov/25503938,2014-12-15,en,amino acid sequence protein affects structure ...,enzymologists distinguish binding (kd) and cat...,unknown
1,https://openalex.org/W3016143870,Space Radiation Biology for âLiving in Spaceâ,https://doi.org/10.1155/2020/4703286,space travel has advanced significantly over t...,2020-01-01,Satoshi Furukawa; Aiko Nagamatsu; Mitsuru Neno...,BioMed Research International,https://doi.org/10.1155/2020/4703286,NaN,https://pubmed.ncbi.nlm.nih.gov/32337251,2020-01-01,en,space travel advanced significantly last six d...,future research that furthers our understandin...,future research


In [146]:
print(len(df2))


# Drop duplicates only on non-null values

print(len(df2))
df2 = df2.drop_duplicates(subset=['openalex_id'], keep='first')
print(f"After deduplicating on openalex_id: {len(df2)}")

cols_to_dedup = ['doi', 'pmcid', 'pubmed_id']

for col in cols_to_dedup:
    df_with_val = df2[df2[col].notna()].drop_duplicates(subset=[col], keep='first')
    df_without_val = df2[df2[col].isna()]
    df2 = pd.concat([df_with_val, df_without_val]).reset_index(drop=True)
    print(f"After deduplicating on {col}: {len(df2)}")

12816
12816
After deduplicating on openalex_id: 12815
After deduplicating on doi: 12803
After deduplicating on pmcid: 12803
After deduplicating on pubmed_id: 12803


In [147]:
#docs = df2[df2['clean_text'].notna()].clean_text.to_list() 
# OR IF OVER 70,000 or so are left: 
docs = df2[df2['sentences'].notna()].clean_text.to_list()

In [148]:
docs[2]

print(len(docs))

12803


# **Topic Modeling**




In [ ]:
from bertopic import BERTopic


# Inject it into your BERTopic initialization
topic_model = BERTopic(
    language="english", # OR language="multilingual"
    calculate_probabilities=True, 
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

2026-05-26 21:22:47,967 - BERTopic - Embedding - Transforming documents to embeddings.


Batches: 100%|██████████| 401/401 [02:04<00:00,  3.23it/s]
2026-05-26 21:24:54,497 - BERTopic - Embedding - Completed ✓
2026-05-26 21:24:54,499 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-26 21:24:55,400 - BERTopic - Dimensionality - Completed ✓
2026-05-26 21:24:55,401 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-26 21:25:05,013 - BERTopic - Cluster - Completed ✓
2026-05-26 21:25:05,016 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-26 21:25:06,120 - BERTopic - Representation - Completed ✓


## Extracting Topics


In [ ]:
freq = topic_model.get_topic_info(); freq.head(15) # -1 should be igonored (filler words)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4563,-1_google_scholar_pubmed_species,"[google, scholar, pubmed, species, data, scopu...",[cells building blocks tissues organs organism...
1,0,197,0_cancer_tumor_angiomyolipoma_cells,"[cancer, tumor, angiomyolipoma, cells, melanom...",[despite recent notable progress medical scien...
2,1,176,1_walls_wall_aba_auxin,"[walls, wall, aba, auxin, arabidopsis, cell, p...",[grand challenge coming decades feed rapidly g...
3,2,151,2_microbial_water_microorganisms_bacteria,"[microbial, water, microorganisms, bacteria, s...",[past years since inception microbial biotechn...
4,3,142,3_health_women_risk_covid,"[health, women, risk, covid, mortality, age, d...",[widely thought lifespans increasing globally ...
5,4,133,4_students_education_learning_student,"[students, education, learning, student, gende...",[covid pandemic impacted education levels vari...
6,5,132,5_mitochondrial_mitochondria_membrane_import,"[mitochondrial, mitochondria, membrane, import...",[small tim proteins mitochondrial intermembran...
7,6,116,6_bioh_protein_structural_mass,"[bioh, protein, structural, mass, proteins, pr...",[describe use antibody based proteomics involv...
8,7,113,7_lung_sp_dcs_iai,"[lung, sp, dcs, iai, ifn, il, mice, pulmonary,...",[research molecular markers functions dendriti...
9,8,110,8_brain_astrocytes_cns_hiv,"[brain, astrocytes, cns, hiv, oatp, microglia,...",[past decades central nervous system cns perip...


In [118]:
len(freq)

179

In [119]:
topic_model.get_topic(0)  # Select the most frequent topic

[('cancer', np.float64(0.032321445792297676)),
 ('tumor', np.float64(0.026813975536218695)),
 ('angiomyolipoma', np.float64(0.012145602321261325)),
 ('cells', np.float64(0.011725575964960282)),
 ('melanoma', np.float64(0.011204970168283262)),
 ('breast', np.float64(0.010616707433602308)),
 ('tumors', np.float64(0.0102794730626623)),
 ('renal', np.float64(0.009802828908462872)),
 ('metastasis', np.float64(0.00977074023632018)),
 ('flt', np.float64(0.00921903745166727))]

## Visualize Topics

NOTE CONSIDER: (https://github.com/cpsievert/LDAvis):

In [123]:
fig = topic_model.visualize_topics()
topic_model.visualize_topics()
fig.write_html(
    "topics_visualization.html",
    config={'responsive': True, 'displayModeBar': True}
)

In [122]:
fig_2 =topic_model.visualize_hierarchy(top_n_topics=100)
topic_model.visualize_hierarchy(top_n_topics=100)
fig_2.write_html("topics_hierarchy.html")

## Visualize Terms

In [124]:
topic_model.visualize_barchart(top_n_topics=10)

In [125]:
topic_model.visualize_term_rank()

# **Topic Representation**
After having created the topic model, you might not be satisfied with some of the parameters you have chosen. Fortunately, BERTopic allows you to update the topics after they have been created.

This allows for fine-tuning the model to your specifications and wishes.

## Update Topics
When you have trained a model and viewed the topics and the words that represent them,
you might not be satisfied with the representation. Perhaps you forgot to remove
stopwords or you want to try out a different `n_gram_range`. We can use the function `update_topics` to update
the topic representation with new parameters for `c-TF-IDF`:


## Topic Reduction
We can also reduce the number of topics after having trained a BERTopic model. The advantage of doing so,
is that you can decide the number of topics after knowing how many are actually created. It is difficult to
predict before training your model how many topics that are in your documents and how many will be extracted.
Instead, we can decide afterwards how many topics seems realistic:





In [ ]:
new_topics, new_probs = topic_model.reduce_topics(docs, topics, probs, nr_topics=50)

TypeError: BERTopic.reduce_topics() got multiple values for argument 'nr_topics'

In [ ]:
freq = topic_model.get_topic_info(); freq.head(15)

,Topic,Count,Name
0,-1,7581,-1_the_and_of_to
1,0,256,0_syndrome_respiratory_acute_severe
2,1,248,1_telehealth_telemedicine_care_to
3,2,224,2_thrombosis_patients_venous_vte
4,3,218,3_pregnancy_pregnant_women_neonates
5,4,216,4_cardiac_myocarditis_myocardial_patients
6,5,177,5_neurological_gbs_brain_cognitive
7,6,158,6_surgery_surgical_elective_pandemic
8,7,149,7_ecmo_ventilation_patients_respiratory
9,8,146,8_model_the_parameters_models


In [ ]:
topic_model.visualize_topics()

In [11]:
topic_model.visualize_barchart(top_n_topics=20)

In [12]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [ ]:
len(new_topics),len(topics)

(12853, 12853)

In [ ]:
df2["topics"] = topics
df2["new_topics"] = new_topics

In [ ]:
!ls

'articles_with_sentences and words1.csv'   drive   sample_data


In [ ]:
df2.to_csv("articles_with_topics.csv",index=None)

# Analysis 

In [ ]:
import pandas as pd
df2 = pd.read_csv("articles_with_topics.csv")

In [ ]:
df2.columns

Index(['cord_uid', 'title', 'doi', 'abstract', 'publish_time', 'authors',
       'journal', 'doi.1', 'pmcid', 'pubmed_id', 'publish_time_new', 'lang',
       'sentences', 'words', 'topics', 'new_topics'],
      dtype='object')

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def ubt(x):
    word_data = x
    #word_data = "lime soda is the best selling item in fast food stores"

    # load nltk's stop word list
    stop_words = list(stopwords.words('english'))
    # extend the stop words list
    #stop_words.extend(["best", "selling", "item", "fast"])

    # tokenise the string and remove stop words
    word_tokens = word_tokenize(word_data)
    clean_word_data = [w for w in word_tokens if not w.lower() in stop_words]

    # get bigrams
    bigrams_list = [" ".join(item) for item in nltk.bigrams(clean_word_data)]
    #print(bigrams_list)

    # get trigrams
    trigrams_list = [" ".join(item) for item in nltk.trigrams(clean_word_data)]
    #print(trigrams_list)
    lst = []
    lst.extend(clean_word_data)
    lst.extend(bigrams_list)
    lst.extend(trigrams_list)
    from collections import Counter
    cnt = Counter(lst)
    del cnt["."]
    return cnt.most_common(50)

In [ ]:
import re
dst =""
for i in range(191):
    st = " ".join(df2[df2.topics==i]["sentences"].tolist())
    st = re.sub(r'[^a-zA-Z0-9. ]', '', st)
    dst+="topic-"+str(i)
    dst +="\n"
    dst += str(ubt(st))
    dst +="\n"

In [ ]:
import re
dst1 =""
for i in range(50):
    st = " ".join(df2[df2.new_topics==i]["sentences"].tolist())
    st = re.sub(r'[^a-zA-Z0-9. ]', '', st)
    dst1+="topic-"+str(i)
    dst1 +="\n"
    dst1 += str(ubt(st))
    dst1 +="\n"

In [ ]:
dst += "\n\n\n *********************Reduced Topics*********************\n\n\n"
dst +=dst1
print(dst)

topic-0
[('telehealth', 184), ('care', 181), ('telemedicine', 148), ('patients', 147), ('pandemic', 135), ('covid19', 121), ('research', 117), ('use', 94), ('future', 90), ('covid19 pandemic', 83), ('health', 79), ('services', 67), ('patient', 67), ('access', 55), ('needed', 54), ('studies', 52), ('known', 46), ('however', 45), ('future research', 45), ('little', 44), ('little known', 43), ('study', 42), ('unknown', 40), ('may', 40), ('satisfaction', 40), ('inperson', 37), ('. future', 37), ('. however', 34), ('among', 33), ('using', 32), ('healthcare', 32), ('. research', 32), ('technology', 31), ('clinical', 30), ('visits', 30), ('impact', 29), ('pandemic .', 29), ('delivery', 28), ('outcomes', 28), ('health care', 28), ('care .', 28), ('medical', 25), ('barriers', 25), ('implementation', 25), ('practice', 25), ('interventions', 24), ('improve', 23), ('review', 22), ('research needed', 22), ('pediatric', 21)]
topic-1
[('covid19', 301), ('patients', 231), ('risk', 90), ('unknown', 85)

In [ ]:
with open('Topics Keywords with frequencies.txt', 'w') as f:
    f.write(dst)

# After Manually Labelled

In [ ]:
import pandas as pd
df = pd.read_csv("articles_with_topics.csv")
df.head(3)

,cord_uid,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,sentences,words,topics,new_topics
0,azdtbnqj,Combined Hyperglycemic Hyperosmolar Syndrome a...,10.1155/2021/6429710,although most children with coronavirus diseas...,2021-11-27,"Tseng, Yu Shan; Tilford, Bradley; Sethuraman, ...",Case Rep Crit Care,10.1155/2021/6429710,PMC8627355,34791286.0,2021-11-27,en,"as the pandemic continues, clinicians should ...",further studies,116,-1
1,k8nbgxsf,Evidences suggesting a possible role of Vitami...,10.4103/ijp.ijp_654_20,the severe acute respiratory syndrome coronavi...,2021-11-24,"Singh, Shruti; Singh, C. M.; Ranjan, Alok; Kum...",Indian J Pharmacol,10.4103/ijp.ijp_654_20,PMC8641745,34854410.0,2021-11-24,en,while some risk factors such as the presence ...,further research,-1,-1
2,t3tlj38t,Predictors to Use Mobile Apps for Monitoring C...,10.2196/28416,ehealth apps have been recognized as a valuab...,2021-12-20,"Jansen-Kosterink, Stephanie; Hurmuz, Marian; d...",JMIR Form Res,10.2196/28416,PMC8691407,34818210.0,2021-12-20,en,ehealth apps have been recognized as a valuab...,unknown,-1,-1


In [ ]:
len(df)

12853

In [ ]:
tp = {
    -1:"Unlabelled",
    0:"Unlabelled",
    1:"TeleHealth",
    2: "Thrombosis",
    3:"Maternal & Neonatal (Pregnancy)",
    4:"Cardiovascular Complications",
    5: "Neurological Complications",
    6: "Surgical Considerations",
    7:"Respiratory (Pulmonary) Complications",
    8 :"Prediction Models",
    9: "Vaccination Efficacy & Safety",
    10:"Mental Health",
    11:"Education",
    12:"Healthcare workers’ Mental Health",
    13:"Vaccination Hesitancy",
    14:"Vaccination Hesitancy",
    15:"Respiratory (Pulmonary) Complications",
    16:"Origin of COVID-19",
    17:"Angiotensin Converting Enzyme (ACE)",
    18:"Media & Communication",
    19:"Origin of COVID-19",
    20:"Diabetes",
    21:"Origin of COVID-19",
    22:"General",
    23:"Organ Transplantation",
    24:"Treatment of COVID-19",
    25:"Mortality",
    26: "Dietary Supplementation",
    27:"General",
    28:"General",
    29: "Obesity",
    30:"Maternal & Neonatal (Pregnancy)",
    31: "Pediatrics",
    32: "Transmission",
    33: "Hepatic Complications",
    34: "Herbal Medicine",
    35: "Public Policies & Precaution Measures",
    36: "Ethnicity",
    37: "Cancer",
    38:"General",
    39: "Food Security",
    40: "Wastewater Surveillance",
    41:"Vaccination Hesitancy",
    42:"General",
    43: "Dental & Oral Health",
    44: "Face Mask",
    45:"TeleHealth",
    46: "Olfactory (Smell & Taste)",
    47: "Emerging Variants",
    48: "Support System",
    49: "Immune System"
}

In [ ]:
df["Topic Labels"]=df["new_topics"].apply(lambda x: tp[x])

In [ ]:
df.groupby(by=["Topic Labels"])["title"].count()

Topic Labels
Angiotensin Converting Enzyme (ACE)       113
Cancer                                     64
Cardiovascular Complications              216
Dental & Oral Health                       59
Diabetes                                   95
Dietary Supplementation                    80
Education                                 136
Emerging Variants                          56
Ethnicity                                  65
Face Mask                                  58
Food Security                              62
General                                   370
Healthcare workers’ Mental Health         136
Hepatic Complications                      70
Herbal Medicine                            66
Immune System                              52
Maternal & Neonatal (Pregnancy)           294
Media & Communication                     109
Mental Health                             138
Mortality                                  87
Neurological Complications                177
Obesity              